# init data

## datasets

In [8]:
import pandas as pd

strongs_df = pd.read_csv("strongs.csv")
lv65_df = pd.read_csv("latvian_full65.csv")
import unicodedata
def raccs_common(text):
    d = {
        #ord('\N{COMBINING ACUTE ACCENT}'):None,
        ord('’'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('‘'): None,  # LEFT SINGLE QUOTATION MARK
        ord('“'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('”'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,  # LEFT SINGLE QUOTATION MARK
        ord(']'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('-'): None,   # HYPHEN-MINUS
        ord('’'): None,   # HYPHEN-MINUS
        ord('⧼'): None,  # LEFT WHITE ANGLE BRACKET
        ord('⧽'): None,  # RIGHT WHITE ANGLE BRACKET
        ord('*'): None,   # ASTERISK
        ord('⇔'): None,  # LEFT RIGHT DOUBLE ARROW
        ord('〉'): None,  # GREATER-THAN SIGN
        ord('〈'): None,  # LESS-THAN SIGN
        ord('‿'): None,  # LOW LINE
        ord('«'): None,  # LEFT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('»'): None,  # RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('‹'): None,  # SINGLE LEFT-POINTING ANGLE QUOTATION MARK
        ord('›'): None,  # SINGLE RIGHT-POINTING ANGLE QUOTATION MARK
        ord('('): None,  # LEFT PARENTHESIS
        ord(')'): None,  # RIGHT PARENTHESIS
        ord('-') : None,  # HYPHEN-MINUS
        ord(';') : None,  # SEMICOLON
        }
    return unicodedata.normalize('NFC', text).translate(d)
print(strongs_df["form"].apply(raccs_common).head())

0       Παῦλος
1       κλητὸς
2    ἀπόστολος
3      Χριστοῦ
4        Ἰησοῦ
Name: form, dtype: str


## select data

In [27]:
import pandas as pd
import time
from pathlib import Path
import json

def prepare_grouped(strongs_df, lv_df):
    strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
    lv_g = lv_df.groupby(["book", "chapter", "verse"])
    return strongs_g, lv_g

import os
import json
import re
from pathlib import Path
from collections import Counter

def validate_latvian_usage(verse_text, mappings):
    """
    Checks if all words in the Latvian text are present in the mappings.
    Allows for repetitions.
    """
    # Tokenize Latvian text (remove punctuation and lowercase)
    text_words = re.findall(r'\b\w+\b', verse_text)#.lower())
    
    # Collect all words used in mappings
    mapped_words = []
    for m in mappings:
        if m.get('latvian'):
            for word in m['latvian']:
                mapped_words.extend(re.findall(r'\b\w+\b', word))#.lower()))
    if m.get("leftover_latvian"):
        for word in m["leftover_latvian"]:
            mapped.words.extend(re.findall(r'\bw+\b', word))
    
    text_counts = Counter(text_words)
    mapped_counts = Counter(mapped_words)

    # Check if any word from text is missing or under-represented in mappings
    missing = []
    for word, count in text_counts.items():
        if mapped_counts[word] < count:
            for i in range (count-mapped_counts[word]):
                missing.append(word)
            #missing.append(f"'{word}' (found {mapped_counts[word]}/{count})")
            
    return missing

def split_chapter_json(book, chapter_num, strongs_g, lv_g):
    # Path to the source chapter file: bible/luke/1.json
    source_path = Path("bible") / book.lower() / f"{chapter_num}.json"
    
    if not source_path.exists():
        print(f"Source {source_path} not found. Skipping.")
        return

    with open(source_path, "r", encoding="utf-8") as f:
        verses_list = json.load(f)

    # Create the destination directory: bible/luke/1/
    dest_dir = Path("bible") / book.lower() / str(chapter_num)
    dest_dir.mkdir(parents=True, exist_ok=True)

    for i, verse_data in enumerate(verses_list):
        key = (book, int(chapter_num), int(i+1))

        # 1. Get Reference Data
        if key not in strongs_g.groups or key not in lv_g.groups:
            print(f"⚠️ Missing reference data for {key}. Skipping.")
            continue

        ref_strongs = strongs_g.get_group(key).sort_values("word")
        ref_latvian_text = lv_g.get_group(key).iloc[0]["text"]

        # 2. Validation: Greek Word Count and Order
        mapping_greek_words = verse_data.get('greek_words', [])
        if len(ref_strongs) != len(mapping_greek_words):
            print(f"❌ Greek word count mismatch at {key}: Ref {len(ref_strongs)} vs JSON {len(mapping_greek_words)}")
            #continue

        # 3. Validation: Latvian Usage
        latvian_missing = validate_latvian_usage(ref_latvian_text, mapping_greek_words)
        if latvian_missing:
            print(f"⚠️ Latvian mapping incomplete at {key}: Missing {', '.join(latvian_missing)}")
            # Optional: continue/break based on how strict you want to be

        # 4. Construct the individual verse object
        # Note: We include the 'locus' as your build_verse_object expects it
        leftover_latvian = verse_data.get("leftover_latvian", [])
        leftover_latvian.extend(latvian_missing)
        verse_output = {
            "locus": f"{book}_{chapter_num}_{i+1}",
            "greek_words": mapping_greek_words,
            "leftover_latvian": leftover_latvian
        }

        # 5. Write to File: bible/luke/1/luke_1_1.txt
        file_name = f"{book}_{chapter_num}_{i+1}.txt"
        with open(dest_dir / file_name, "w", encoding="utf-8") as out_f:
            json.dump(verse_output, out_f, ensure_ascii=False, indent=4)

    print(f"✅ Successfully split {source_path} into {dest_dir}")

# --- Execution ---
# Grouped dataframes from your existing code
strongs_g, lv_g = prepare_grouped(strongs_df, lv65_df)

# doit()

In [36]:
split_chapter_json("luke", 17, strongs_g, lv_g)

⚠️ Latvian mapping incomplete at ('luke', 17, 1): Missing Viņš, nenāk, tā
⚠️ Latvian mapping incomplete at ('luke', 17, 2): Missing Tādam, apliktu, to, no
⚠️ Latvian mapping incomplete at ('luke', 17, 3): Missing viņš
⚠️ Latvian mapping incomplete at ('luke', 17, 4): Missing viņš
⚠️ Latvian mapping incomplete at ('luke', 17, 6): Missing un, uz, koku, ar, savām, saknēm, viņš
⚠️ Latvian mapping incomplete at ('luke', 17, 8): Missing viņš, neteiks, un
⚠️ Latvian mapping incomplete at ('luke', 17, 9): Missing viņš, tas
⚠️ Latvian mapping incomplete at ('luke', 17, 12): Missing Te
⚠️ Latvian mapping incomplete at ('luke', 17, 14): Missing Viņš, tos, rādaities, ceļā
⚠️ Latvian mapping incomplete at ('luke', 17, 15): Missing viņš
⚠️ Latvian mapping incomplete at ('luke', 17, 17): Missing tie
⚠️ Latvian mapping incomplete at ('luke', 17, 18): Missing cits
⚠️ Latvian mapping incomplete at ('luke', 17, 19): Missing Viņš
⚠️ Latvian mapping incomplete at ('luke', 17, 20): Missing Viņš, nenāk
⚠️ La